# Bibliotecas

In [44]:
import  pandas                  as pd
import  numpy                   as np
import  plotly.graph_objects    as go
import  plotly.express          as px
import  matplotlib.pyplot       as plt
import  xgboost                 as xgb

from    plotly.subplots         import make_subplots
from    sklearn.preprocessing   import StandardScaler
from    sklearn.metrics         import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    roc_curve,
    auc,
    roc_auc_score
    )
from    sklearn.model_selection             import train_test_split
from    sklearn.ensemble                    import RandomForestClassifier
from    datetime                            import datetime as dt

In [2]:
RANDOM_SEED = 0

# Base cadastral

In [3]:
df_base_cadast = pd.read_csv('data\\base_cadastral.csv',
                             sep=';',
                             encoding='utf-8'
                             )
df_base_cadast.columns = df_base_cadast.columns.str.lower()
df_base_cadast['id_cliente'] = df_base_cadast['id_cliente'].astype(str)

# Cria dummie 1 = PF e 0 = PJ
df_base_cadast['flag_pf'] = df_base_cadast['flag_pf'] .map({'X': 1, np.nan: 0}).astype(int)

# Extrai o primeiro bloco de números da string e converte para float (mantendo NaN se não houver números)
df_base_cadast['ddd'] = (
    df_base_cadast['ddd']
    .str.extract(r'(\d+)', expand=False)
    .astype(float)
)

# Trata CEP e converte para float para modelagem
df_base_cadast['cep_2_dig'] = df_base_cadast['cep_2_dig'].replace({'na': np.nan}).astype(float)

df_base_cadast['data_cadastro'] = pd.to_datetime(df_base_cadast['data_cadastro'])
data_atual = dt.now()
# Cria feature com a diferença entre a data atual e a data de cadastro em anos
df_base_cadast['anos_cliente'] = np.floor(
    (data_atual - df_base_cadast['data_cadastro']) / pd.Timedelta(days=365)
    ).astype('Int64')

df_base_cadast

,id_cliente,data_cadastro,ddd,flag_pf,segmento_industrial,dominio_email,porte,cep_2_dig,anos_cliente
0,1661240395903230676,2013-08-22,99.0,0,Serviços,YAHOO,PEQUENO,65.0,12
1,8274986328479596038,2017-01-25,31.0,0,Comércio,YAHOO,MEDIO,77.0,9
2,345447888460137901,2000-08-15,75.0,0,Serviços,HOTMAIL,PEQUENO,48.0,25
3,1003144834589372198,2017-08-06,49.0,0,Serviços,OUTLOOK,PEQUENO,89.0,8
4,324916756972236008,2011-02-14,88.0,0,Serviços,GMAIL,GRANDE,62.0,15
...,...,...,...,...,...,...,...,...,...
1310,3431426889924624821,2020-08-13,92.0,0,Serviços,HOTMAIL,MEDIO,69.0,5
1311,5288503299611498087,2020-11-03,NaN,0,Comércio,YAHOO,PEQUENO,13.0,5
1312,957773253650890560,2021-07-05,NaN,0,Comércio,GMAIL,MEDIO,20.0,5
1313,6094038865287329652,2021-07-05,NaN,0,Serviços,GMAIL,GRANDE,48.0,5


In [4]:
df_base_cadast.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1315 entries, 0 to 1314
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   id_cliente           1315 non-null   object        
 1   data_cadastro        1315 non-null   datetime64[ns]
 2   ddd                  1078 non-null   float64       
 3   flag_pf              1315 non-null   int64         
 4   segmento_industrial  1232 non-null   object        
 5   dominio_email        1285 non-null   object        
 6   porte                1274 non-null   object        
 7   cep_2_dig            1314 non-null   float64       
 8   anos_cliente         1315 non-null   Int64         
dtypes: Int64(1), datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 93.9+ KB


In [5]:
df_base_cadast.isna().sum()

id_cliente               0
data_cadastro            0
ddd                    237
flag_pf                  0
segmento_industrial     83
dominio_email           30
porte                   41
cep_2_dig                1
anos_cliente             0
dtype: int64

In [6]:
# Identifica se há ids repetidos
if df_base_cadast['id_cliente'].nunique() != df_base_cadast.shape[0]:
    print('Há id repetido')
else:
    print('Não há id repetido')

Não há id repetido


In [7]:
# Histograma da distribuição do tempo de relacionamento em anos
fig_anos = px.histogram(
    df_base_cadast,
    x='anos_cliente',
    nbins=25,
    title='Distribuição do Tempo de Relacionamento dos Clientes',
    labels={
        'anos_cliente': 'Anos de Relacionamento',
        'count': 'Quantidade de Clientes'
        },
    text_auto=True,
    color_discrete_sequence=['#1f77b4']
)

fig_anos.update_layout(
    bargap=0.1,
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(
        title='Qt. Clientes',
        showticklabels=False,
        showgrid=True,
        gridcolor='lightgray')
)

fig_anos.show()

In [8]:
df_base_cadast[['anos_cliente']].describe()

,anos_cliente
count,1315.0
mean,12.460837
std,6.27131
min,5.0
25%,7.0
50%,11.0
75%,15.0
max,25.0


In [9]:
# Agrupando os dados para o gráfico de barras
df_porte_segmento = (
    df_base_cadast.groupby(['segmento_industrial', 'porte'], dropna=False)
    .size()
    .reset_index(name='quantidade')
)

# Gráfico de barras agrupadas por Segmento e Porte
fig_segmento = px.bar(
    df_porte_segmento,
    x='segmento_industrial',
    y='quantidade',
    color='porte',
    barmode='group',
    title='Volume de Clientes por Segmento Industrial e Porte',
    labels={
        'segmento_industrial': 'Segmento Industrial',
        'quantidade': 'Quantidade de Clientes',
        'porte': 'Porte da Empresa'
    },
    text_auto=True,
    color_discrete_sequence=['#1f77b4']
)

fig_segmento.update_layout(
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, showticklabels=False, gridcolor='lightgray'),
    legend_title_text='Porte'
)

fig_segmento.show()

In [10]:
flag_pf_0 = df_base_cadast[df_base_cadast['flag_pf'] == 0].shape[0]
flag_pf_1 = df_base_cadast[df_base_cadast['flag_pf'] == 1].shape[0]

total_clientes = df_base_cadast.shape[0]

pct_pf_0 = np.round((flag_pf_0 / total_clientes) * 100, 2)
pct_pf_1 = np.round((flag_pf_1 / total_clientes) * 100, 2)

# Criando um DataFrame auxiliar para o gráfico de proporção
df_pf_counts = pd.DataFrame({
    'tipo_cliente': ['PJ', 'PF'],
    'quantidade': [pct_pf_0, pct_pf_1]
})

fig_pf = px.bar(
    df_pf_counts,
    x='tipo_cliente',
    y='quantidade',
    title='Proporção de Clientes: Pessoa Física vs Pessoa Jurídica',
    labels={
        'tipo_cliente': 'Tipo de Cliente',
        'quantidade': 'Qt. Clientes'
    },
    text_auto=True,
    color_discrete_sequence=['#1f77b4']
)

fig_pf.update_layout(
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(
        showgrid=True,
        showticklabels=False,
        gridcolor='lightgray',
        title='% Clientes'
    ),
    showlegend=False
)

fig_pf.show()

## Análise dos clientes por Estado

In [11]:
# Relação entre os dois primeiros digitos do CEP e os estados brasileiros
ceps_uf = {
    "10": "SP", "11": "SP", "12": "SP", "13": "SP", "14": "SP", "15": "SP", "16": "SP", "17": "SP", "18": "SP", "19": "SP",
    "20": "RJ", "21": "RJ", "22": "RJ", "23": "RJ", "24": "RJ",
    "25": "RJ", "26": "RJ", "27": "RJ", "28": "RJ", "29": "RJ",
    "30": "MG", "31": "MG", "32": "MG", "33": "MG", "34": "MG", "35": "MG", "36": "MG", "37": "MG", "38": "MG", "39": "MG",
    "40": "BA", "41": "BA", "42": "BA", "43": "BA", "44": "BA", "45": "BA", "46": "BA", "47": "BA", "48": "BA", "49": "BA",
    "50": "MS", "51": "MS", "52": "GO", "53": "DF", "54": "RS", "55": "RS", "56": "RS", "57": "AL", "58": "PB", "59": "RN",
    "60": "CE", "61": "CE", "62": "CE", "63": "CE", "64": "CE", "65": "CE", "66": "CE", "67": "CE", "68": "PA", "69": "RO",
    "70": "DF", "71": "DF", "72": "DF", "73": "DF", "74": "DF", "75": "DF", "76": "DF", "77": "DF", "78": "MT", "79": "SE",
    "80": "PR", "81": "PR", "82": "PR", "83": "PR", "84": "PR", "85": "PR", "86": "PR", "87": "PR", "88": "SC", "89": "SC",
    "90": "RS", "91": "RS", "92": "RS", "93": "RS", "94": "RS", "95": "RS", "96": "AP", "97": "AM", "98": "MA", "99": "MA"
}

In [12]:
df_ceps_brasil = pd.DataFrame(list(ceps_uf.items()), columns=['cep_2_dig', 'uf'])
df_ceps_brasil['cep_2_dig'] = df_ceps_brasil['cep_2_dig'].astype('Int64')
df_ceps_brasil

,cep_2_dig,uf
0,10,SP
1,11,SP
2,12,SP
3,13,SP
4,14,SP
...,...,...
85,95,RS
86,96,AP
87,97,AM
88,98,MA


In [13]:
df_base_cadast_uf = df_base_cadast.merge(df_ceps_brasil, on='cep_2_dig', how='left')
df_base_cadast_uf

,id_cliente,data_cadastro,ddd,flag_pf,segmento_industrial,dominio_email,porte,cep_2_dig,anos_cliente,uf
0,1661240395903230676,2013-08-22,99.0,0,Serviços,YAHOO,PEQUENO,65.0,12,CE
1,8274986328479596038,2017-01-25,31.0,0,Comércio,YAHOO,MEDIO,77.0,9,DF
2,345447888460137901,2000-08-15,75.0,0,Serviços,HOTMAIL,PEQUENO,48.0,25,BA
3,1003144834589372198,2017-08-06,49.0,0,Serviços,OUTLOOK,PEQUENO,89.0,8,SC
4,324916756972236008,2011-02-14,88.0,0,Serviços,GMAIL,GRANDE,62.0,15,CE
...,...,...,...,...,...,...,...,...,...,...
1310,3431426889924624821,2020-08-13,92.0,0,Serviços,HOTMAIL,MEDIO,69.0,5,RO
1311,5288503299611498087,2020-11-03,NaN,0,Comércio,YAHOO,PEQUENO,13.0,5,SP
1312,957773253650890560,2021-07-05,NaN,0,Comércio,GMAIL,MEDIO,20.0,5,RJ
1313,6094038865287329652,2021-07-05,NaN,0,Serviços,GMAIL,GRANDE,48.0,5,BA


In [14]:
df_uf_pct = (
    df_base_cadast_uf['uf']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .reset_index(name='percentual')
)

fig_pf = px.bar(
    df_uf_pct,
    x='uf',
    y='percentual',
    title='Proporção de Clientes por Estado',
    labels={
        'uf': 'Estado',
        'percentual': '%% de Clientes'
    },
    text_auto=True,
    color_discrete_sequence=['#1f77b4']
)

fig_pf.update_layout(
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(
        showgrid=True,
        showticklabels=False,
        gridcolor='lightgray',
        title='% Clientes'
    ),
    showlegend=False
)

fig_pf.show()

# Base info

In [15]:
df_base_info = pd.read_csv('data\\base_info.csv',
                           sep=';',
                           encoding='utf-8'
                           )
df_base_info.columns = df_base_info.columns.str.lower()
df_base_info['id_cliente'] = df_base_info['id_cliente'].astype(str)
df_base_info

,id_cliente,safra_ref,renda_mes_anterior,no_funcionarios
0,1661240395903230676,2018-09,16913.0,NaN
1,8274986328479596038,2018-09,106430.0,141.0
2,345447888460137901,2018-09,707439.0,99.0
3,1003144834589372198,2018-09,239659.0,96.0
4,324916756972236008,2018-09,203123.0,103.0
...,...,...,...,...
24396,705648002974742140,2021-12,278663.0,105.0
24397,4993499380140734678,2021-12,156968.0,140.0
24398,4614484019183480654,2021-12,292698.0,121.0
24399,1299146298565441811,2021-12,106180.0,121.0


In [16]:
df_base_info.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24401 entries, 0 to 24400
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_cliente          24401 non-null  object 
 1   safra_ref           24401 non-null  object 
 2   renda_mes_anterior  23684 non-null  float64
 3   no_funcionarios     23149 non-null  float64
dtypes: float64(2), object(2)
memory usage: 762.7+ KB


## Tratamento de dados nulos

In [17]:
df_base_info.isna().sum()

id_cliente               0
safra_ref                0
renda_mes_anterior     717
no_funcionarios       1252
dtype: int64

In [18]:
df_base_info_nulos = df_base_info[['renda_mes_anterior', 'no_funcionarios']].isnull().sum()
perc_df_base_info_nulos = (df_base_info_nulos / len(df_base_info) * 100).round(2)

resumo_df_base_info_nulos = pd.DataFrame({'qtd_nulos': df_base_info_nulos, 'pct_nulos': perc_df_base_info_nulos})
resumo_df_base_info_nulos

,qtd_nulos,pct_nulos
renda_mes_anterior,717,2.94
no_funcionarios,1252,5.13


In [19]:
# Nulos são concentrados em poucos clientes ou espalhados?
clientes_com_null_renda = df_base_info.loc[df_base_info['renda_mes_anterior'].isnull(), 'id_cliente'].nunique()
clientes_com_null_func = df_base_info.loc[df_base_info['no_funcionarios'].isnull(), 'id_cliente'].nunique()

print(f'Clientes distintos com renda_mes_anterior nula: {clientes_com_null_renda} (de {df_base_info["id_cliente"].nunique()} clientes)')
print(f'Clientes distintos com no_funcionarios nula: {clientes_com_null_func} (de {df_base_info["id_cliente"].nunique()} clientes)')
print(f'Linhas com AMBAS as colunas nulas: {df_base_info["renda_mes_anterior"].isnull().mul(df_base_info["no_funcionarios"].isnull()).sum()}')

Clientes distintos com renda_mes_anterior nula: 475 (de 1336 clientes)
Clientes distintos com no_funcionarios nula: 681 (de 1336 clientes)
Linhas com AMBAS as colunas nulas: 33


In [20]:
# Os nulos se concentram em alguma safra específica (ex: um mês com problema de coleta)?
nulos_por_safra = (
    df_base_info
    .groupby('safra_ref')[['renda_mes_anterior', 'no_funcionarios']]
    .apply(lambda x: x.isnull().mean()*100)
    )

# Transforma o DataFrame para o formato longo, adequado para plot
nulos_por_safra_long = nulos_por_safra.reset_index().melt(
    id_vars='safra_ref', var_name='variavel', value_name='pct_nulo'
)

fig_nulos = px.line(
    nulos_por_safra_long,
    x='safra_ref',
    y='pct_nulo',
    color='variavel',
    markers=True,
    title='% de Valores Nulos por Safra',
    labels={
        'safra_ref': 'Safra',
        'pct_nulo': '% Nulo',
        'variavel': 'Variável'
    }
)

fig_nulos.update_layout(
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray', tickangle=90),
    yaxis=dict(showgrid=True, gridcolor='lightgray')
)

fig_nulos.show()

In [21]:
df_base_info.columns

Index(['id_cliente', 'safra_ref', 'renda_mes_anterior', 'no_funcionarios'], dtype='object')

**Conclusão**
- Os nulos representam uma fração pequena da base (3% em `renda_mes_anterior` e 5% em `no_funcionarios`).
- Não há concentração forte em uma safra específica — os nulos parecem distribuídos ao longo do tempo, sugerindo ausência de informação pontual (ex: cliente não enviou/atualizou o dado naquele mês) e não uma falha de coleta em um mês específico.

In [22]:
df_base_info_tratado = df_base_info.sort_values(['id_cliente', 'safra_ref']).copy()

for col in ['renda_mes_anterior', 'no_funcionarios']:
    # Preenchimento por cliente (ffill + bfill cobre início/fim de série)
    df_base_info_tratado[col] = df_base_info_tratado.groupby('id_cliente')[col].transform(lambda s: s.ffill().bfill())

# Trata cliente sem nenhum valor histórico com mediana da safra
for col in ['renda_mes_anterior', 'no_funcionarios']:
    df_base_info_tratado[col] = df_base_info_tratado.groupby('safra_ref')[col].transform(lambda s: s.fillna(s.median()))

print('Nulos restantes após tratamento:')
print(df_base_info_tratado[['renda_mes_anterior', 'no_funcionarios']].isnull().sum())

Nulos restantes após tratamento:
renda_mes_anterior    0
no_funcionarios       0
dtype: int64


In [23]:
# Comparativo antes x depois
comparativo_base_info = pd.concat([
    (
        df_base_info[['renda_mes_anterior', 'no_funcionarios']]
        .describe()
        .round(2)
        .add_suffix('_antes')
    ),
    (
        df_base_info_tratado[['renda_mes_anterior', 'no_funcionarios']]
        .describe()
        .round(2)
        .add_suffix('_depois')
        )
], axis=1)

comparativo_base_info[['renda_mes_anterior_antes', 'renda_mes_anterior_depois',
             'no_funcionarios_antes', 'no_funcionarios_depois']]

,renda_mes_anterior_antes,renda_mes_anterior_depois,no_funcionarios_antes,no_funcionarios_depois
count,23684.00,24401.00,23149.00,24401.00
mean,288751.40,288655.36,117.80,117.78
std,211594.79,211375.08,21.46,21.44
min,105.00,105.00,0.00,0.00
25%,133866.25,133875.00,106.00,106.00
50%,240998.50,241184.00,118.00,118.00
75%,392501.75,392241.00,131.00,131.00
max,1682759.00,1682759.00,198.00,198.00


**Conclusão**
- Como as estatísticas descritivas não sofreram alterações em relação ao tratamento, os dados estão limpos e preparados para análise

In [24]:
df_evolucao_renda_mensal = (
    df_base_info_tratado
    .groupby('safra_ref')['renda_mes_anterior']
    .mean()
    .reset_index(name='renda_media')
    )

fig_nulos = px.line(
    df_evolucao_renda_mensal,
    x='safra_ref',
    y='renda_media',
    markers=True,
    title='Evolução da Renda Média Mensal dos Clientes por Safra',
    color_discrete_sequence=['#1f77b4'],
    labels={
        'safra_ref': 'Safra',
        'renda_media': 'Renda Média'
    }
)

fig_nulos.update_layout(
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray', tickangle=90),
    yaxis=dict(showgrid=True, gridcolor='lightgray')
)

fig_nulos.show()

In [25]:
df_renda_media_clientes = (
    df_base_info
    .groupby("id_cliente")["renda_mes_anterior"]
    .mean()
    .reset_index(name="renda_media")
)

fig_renda_media = px.histogram(
    df_renda_media_clientes,
    x='renda_media',
    nbins=100,
    title='Distribuição da Renda Média dos Clientes',
    labels={
        'renda_media': 'Renda Média',
        'count': 'Quantidade de Clientes'
    },
    color_discrete_sequence=['#1f77b4']
)

fig_renda_media.update_layout(
    bargap=0.1,
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(
        title='Qt. Clientes',
        showgrid=True,
        gridcolor='lightgray')
)

fig_renda_media.show()

In [26]:
fig_box_renda = px.box(
    df_renda_media_clientes,
    x='renda_media',
    title='Distribuição da Renda Média dos Clientes',
    labels={
        'renda_media': 'Renda Média',
        'count': 'Quantidade de Clientes'
    },
    color_discrete_sequence=['#1f77b4']
)

fig_box_renda.update_layout(
    bargap=0.1,
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(
        title='Qt. Clientes',
        showgrid=True,
        gridcolor='lightgray')
)

fig_box_renda.show()

In [27]:
df_evolucao_no_funcionarios = (
    df_base_info_tratado
    .groupby('safra_ref')['no_funcionarios']
    .mean()
    .reset_index(name='no_funcionarios')
    )

fig_nulos = px.line(
    df_evolucao_no_funcionarios,
    x='safra_ref',
    y='no_funcionarios',
    markers=True,
    title='Evolução do Número Médio de Funcionários por Safra',
    color_discrete_sequence=['#1f77b4'],
    labels={
        'safra_ref': 'Safra',
        'no_funcionarios': 'Número de Funcionários'
    }
)

fig_nulos.update_layout(
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray', tickangle=90),
    yaxis=dict(showgrid=True, gridcolor='lightgray')
)

fig_nulos.show()

# Base pagamentos

In [28]:
df_base_pagamentos = pd.read_csv('data\\base_pagamentos_desenvolvimento.csv',
                                 sep=';',
                                 encoding='utf-8'
                                 )
df_base_pagamentos.columns = df_base_pagamentos.columns.str.lower()
df_base_pagamentos['id_cliente'] = df_base_pagamentos['id_cliente'].astype(str)
df_base_pagamentos['data_emissao_documento'] = pd.to_datetime(df_base_pagamentos['data_emissao_documento'])
df_base_pagamentos['data_pagamento'] = pd.to_datetime(df_base_pagamentos['data_pagamento'])
df_base_pagamentos['data_vencimento'] = pd.to_datetime(df_base_pagamentos['data_vencimento'])
df_base_pagamentos['dias_atraso'] = (df_base_pagamentos['data_pagamento'] - df_base_pagamentos['data_vencimento']).dt.days

# Cria dummie de inadimplência
df_base_pagamentos['inadimplente'] = (df_base_pagamentos['dias_atraso'] >= 5).astype(int)

df_base_pagamentos

,id_cliente,safra_ref,data_emissao_documento,data_pagamento,data_vencimento,valor_a_pagar,taxa,dias_atraso,inadimplente
0,1661240395903230676,2018-08,2018-08-17,2018-09-06,2018-09-06,35516.41,6.99,0,0
1,1661240395903230676,2018-08,2018-08-19,2018-09-11,2018-09-10,17758.21,6.99,1,0
2,1661240395903230676,2018-08,2018-08-26,2018-09-18,2018-09-17,17431.96,6.99,1,0
3,1661240395903230676,2018-08,2018-08-30,2018-10-11,2018-10-05,1341.00,6.99,6,1
4,1661240395903230676,2018-08,2018-08-31,2018-09-20,2018-09-20,21309.85,6.99,0,0
...,...,...,...,...,...,...,...,...,...
77409,2951563549197799278,2021-06,2021-06-30,2021-07-16,2021-07-16,89980.00,5.99,0,0
77410,5220206408301580591,2021-06,2021-06-30,2021-08-16,2021-08-16,42239.00,5.99,0,0
77411,5860276371789140450,2021-06,2021-06-30,2021-07-16,2021-07-16,20921.50,5.99,0,0
77412,2814790209436551216,2021-06,2021-06-30,2021-07-16,2021-07-16,90231.05,6.99,0,0


In [29]:
# df_base_pagamentos.to_csv('data\\base_pagamentos_tratado.csv', index=False, sep=';', encoding='utf-8')

In [30]:
df_base_pagamentos.drop(columns=['inadimplente']).describe(include=[np.number]).round(2)

,valor_a_pagar,taxa,dias_atraso
count,76244.00,77414.00,77414.00
mean,46590.78,6.79,-0.17
std,46433.93,1.80,25.23
min,0.10,4.99,-2661.00
25%,18765.36,5.99,0.00
50%,34758.70,5.99,0.00
75%,60933.84,6.99,0.00
max,4400000.00,11.99,869.00


In [31]:
df_base_pagamentos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 77414 entries, 0 to 77413
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   id_cliente              77414 non-null  object        
 1   safra_ref               77414 non-null  object        
 2   data_emissao_documento  77414 non-null  datetime64[ns]
 3   data_pagamento          77414 non-null  datetime64[ns]
 4   data_vencimento         77414 non-null  datetime64[ns]
 5   valor_a_pagar           76244 non-null  float64       
 6   taxa                    77414 non-null  float64       
 7   dias_atraso             77414 non-null  int64         
 8   inadimplente            77414 non-null  int64         
dtypes: datetime64[ns](3), float64(2), int64(2), object(2)
memory usage: 5.3+ MB


In [32]:
flag_inadimp_0 = df_base_pagamentos[df_base_pagamentos['inadimplente'] == 0].shape[0]
flag_inadimp_1 = df_base_pagamentos[df_base_pagamentos['inadimplente'] == 1].shape[0]

total_clientes = df_base_pagamentos.shape[0]

pct_inadimp_0 = np.round((flag_inadimp_0 / total_clientes) * 100, 2)
pct_inadimp_1 = np.round((flag_inadimp_1 / total_clientes) * 100, 2)

# Criando um DataFrame auxiliar para o gráfico de proporção
df_inadimp_counts = pd.DataFrame({
    'tipo_cliente': ['Adimplente', 'Inadimplente'],
    'quantidade': [pct_inadimp_0, pct_inadimp_1]
})

fig_inadimp = px.bar(
    df_inadimp_counts,
    x='tipo_cliente',
    y='quantidade',
    title='Proporção de Clientes: Adimplente vs Inadimplente',
    labels={
        'tipo_cliente': 'Tipo de Cliente',
        'quantidade': 'Qt. Clientes'
    },
    text_auto=True,
    color_discrete_sequence=['#1f77b4']
)

fig_inadimp.update_layout(
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(
        showgrid=True,
        showticklabels=False,
        gridcolor='lightgray',
        title='% Clientes'
    ),
    showlegend=False
)

fig_inadimp.show()

**Conclusão**
- Dataset de treino/teste desbalanceado

In [33]:
# Evolução da inadimplência por safra
df_inad_safra = (
    df_base_pagamentos.groupby("safra_ref")["inadimplente"]
      .mean()
      .reset_index(name="inadimplente")
)

df_inad_safra["inadimplente"] *= 100
df_inad_safra["safra_ref"] = df_inad_safra["safra_ref"].astype(str)

fig_inad_safra = px.line(
    df_inad_safra,
    x='safra_ref',
    y='inadimplente',
    markers=True,
    title='Taxa de Inadimplência por Safra',
    color_discrete_sequence=['#1f77b4'],
    labels={
        'safra_ref': 'Safra',
        'inadimplente': '% Inadimplência'
    }
)

fig_inad_safra.update_layout(
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray', tickangle=90),
    yaxis=dict(showgrid=True, gridcolor='lightgray')
)

fig_inad_safra.show()

In [34]:
df_clientes_inad = (
    df_base_pagamentos.groupby("id_cliente")
      .agg(
          qtd_operacoes=("inadimplente", "count"),
          qtd_inad=("inadimplente", "sum")
      )
      .reset_index()
)

df_clientes_inad["tx_inadimplencia"] = (
    (
        df_clientes_inad["qtd_inad"] /
        df_clientes_inad["qtd_operacoes"] * 100
    )
    .round(2)
)

df_clientes_inad.sort_values(
    "tx_inadimplencia",
    ascending=False,
    inplace=True
)

# Filtra clientes com taxa de inadimplência maior que 0 e no mínimo de 5 operações
df_clientes_inad = df_clientes_inad[(df_clientes_inad["tx_inadimplencia"] > 0) & (df_clientes_inad["qtd_operacoes"] >= 5)]

In [35]:
fig_clientes_inadimp = px.bar(
    df_clientes_inad,
    x='id_cliente',
    y='tx_inadimplencia',
    title='Taxa de Inadimplência por Cliente (mínimo de 5 operações)',
    labels={
        'id_cliente': 'Cliente',
        'tx_inadimplencia': 'Taxa de Inadimplência'
        },
    text_auto=True,
    color_discrete_sequence=['#1f77b4']
)

fig_clientes_inadimp.update_layout(
    bargap=0.1,
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(
        showticklabels=False,
        showgrid=True,
        gridcolor='lightgray')
)

fig_clientes_inadimp.show()

In [36]:
df_clientes_inad_merge = df_clientes_inad.merge(df_base_cadast_uf, on='id_cliente', how='left')

fig = px.scatter(
    df_clientes_inad_merge,
    x='anos_cliente',
    y='tx_inadimplencia',
    title='Taxa de Inadimplência por Tempo de Relacionamento do Cliente',
    labels={
        'anos_cliente': 'Anos Cliente',
        'tx_inadimplencia': 'Taxa de Inadimplência'
    },
    color_discrete_sequence=['#1f77b4'],
    trendline="ols",
)

fig.update_layout(
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgray'
    ),
    showlegend=False
)

fig.show() 

**Conclusão**
- Clientes mais antigos parecem ser mais estáveis e menos propensos à inadimplência

## Tratamento de dados nulos

In [37]:
df_base_pagamentos.isna().sum()

id_cliente                   0
safra_ref                    0
data_emissao_documento       0
data_pagamento               0
data_vencimento              0
valor_a_pagar             1170
taxa                         0
dias_atraso                  0
inadimplente                 0
dtype: int64

In [38]:
df_base_pagamentos_nulos = df_base_pagamentos[['valor_a_pagar']].isnull().sum()
perc_df_base_pagamentos_nulos = (df_base_pagamentos_nulos / len(df_base_pagamentos) * 100).round(2)

resumo_df_base_pagamentos_nulos = pd.DataFrame({'qtd_nulos': df_base_pagamentos_nulos, 'pct_nulos': perc_df_base_pagamentos_nulos})
resumo_df_base_pagamentos_nulos

,qtd_nulos,pct_nulos
valor_a_pagar,1170,1.51


In [39]:
valores_a_pagar_nulos_por_safra = (
    df_base_pagamentos
    .groupby('safra_ref')[['valor_a_pagar']]
    .apply(lambda x: x.isnull().mean()*100)
    .reset_index()
    .rename(columns={'valor_a_pagar': 'pct_nulo'})
    )

fig = px.line(
    valores_a_pagar_nulos_por_safra,
    x='safra_ref',
    y='pct_nulo',
    markers=True,
    title='% de Valores a Pagar Nulos por Safra',
    labels={
        'safra_ref': 'Safra',
        'pct_nulo': '% Nulo'
    }
)

fig.update_layout(
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray', tickangle=90),
    yaxis=dict(showgrid=True, gridcolor='lightgray')
)

fig.show()

**Conclusão**
- Os nulos representam uma fração pequena da base (1,5%).
- Não há concentração forte em uma safra específica — os nulos parecem distribuídos ao longo do tempo, sugerindo ausência de informação pontual e não uma falha de coleta em um mês específico.

In [40]:
# Histograma da distribuição do valor a pagar dos clientes
fig_anos = px.histogram(
    df_base_pagamentos,
    x='valor_a_pagar',
    nbins=100,
    title='Distribuição do Valor a Pagar dos Clientes',
    labels={
        'valor_a_pagar': 'Valor a Pagar',
        'count': 'Quantidade de Clientes'
        },
    color_discrete_sequence=['#1f77b4']
)

fig_anos.update_layout(
    bargap=0.1,
    title_x=0.5,
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(
        title='Qt. Clientes',
        showticklabels=True,
        showgrid=True,
        gridcolor='lightgray')
)

fig_anos.show()

In [41]:
# Distribuição é bem assimétrica, por isso usar mediana
mediana_cliente = df_base_pagamentos.groupby('id_cliente')['valor_a_pagar'].transform('median')

df_base_pagamentos_tratado = df_base_pagamentos.copy()

# Cria a flag de imputação baseada na presença ou ausência de um valor para o modelo
df_base_pagamentos_tratado['valor_a_pagar_imputado'] = df_base_pagamentos_tratado['valor_a_pagar'].isnull().astype(int)

# Mediana histórica do próprio cliente (mantém a escala individual dele)
df_base_pagamentos_tratado['valor_a_pagar'] = df_base_pagamentos_tratado['valor_a_pagar'].fillna(mediana_cliente)

# Cliente sem nenhum outro valor recebe a mediana geral da safra
mediana_safra = df_base_pagamentos_tratado.groupby('safra_ref')['valor_a_pagar'].transform('median')
df_base_pagamentos_tratado['valor_a_pagar'] = df_base_pagamentos_tratado['valor_a_pagar'].fillna(mediana_safra)

In [42]:
df_base_pagamentos_tratado.columns

Index(['id_cliente', 'safra_ref', 'data_emissao_documento', 'data_pagamento',
       'data_vencimento', 'valor_a_pagar', 'taxa', 'dias_atraso',
       'inadimplente', 'valor_a_pagar_imputado'],
      dtype='object')

In [43]:
# Comparativo antes x depois
comparativo_base_pagamentos = pd.concat([
    (
        df_base_pagamentos[['valor_a_pagar']]
        .describe()
        .round(2)
        .add_suffix('_antes')
    ),
    (
        df_base_pagamentos_tratado[['valor_a_pagar']]
        .describe()
        .round(2)
        .add_suffix('_depois')
        )
], axis=1)

comparativo_base_pagamentos[['valor_a_pagar_antes', 'valor_a_pagar_depois']]

,valor_a_pagar_antes,valor_a_pagar_depois
count,76244.00,77414.00
mean,46590.78,46543.88
std,46433.93,46229.12
min,0.10,0.10
25%,18765.36,18838.36
50%,34758.70,34764.10
75%,60933.84,60817.29
max,4400000.00,4400000.00


**Conclusão**
- Como as estatísticas descritivas não sofreram alterações em relação ao tratamento, os dados estão limpos e preparados para uso